In [1]:
import pandas as pd
import geopandas as gpd
import matplotlib.pyplot as plt
import plotly.express as px

In [2]:
df_clients = pd.read_csv("df_clients.csv")
df_products_usd = pd.read_csv("df_products_usd.csv")
df_orders_usd = pd.read_csv("df_orders_dolar.csv")
df_product_orders_usd = pd.read_csv("df_product_orders_usd.csv")
df_rates = pd.read_csv("df_rates.csv")

<div style="
    background: linear-gradient(#0c4555);
    padding: 18px 22px;
    border-radius: 10px;
    font-size: 16px;
    font-weight: 500;
    color: #ffffff;
    line-height: 1.6;
    font-family: -apple-system, BlinkMacSystemFont, 'Segoe UI', Roboto, sans-serif;
    border-left: 4px solid #44b5f7;
">
Tablas
</div>

In [3]:
display(df_clients.head(2))
df_clients.shape

,client_id,nombre,telefono,direccion,barrio,lat,lon
0,13,Octavio 1,1.530437e+09,intendente becco 2036,beccar,-34.462121,-58.532741
1,14,Alejandro Grondona,4.747294e+07,almafuerte 1079,san isidro,-33.344373,-60.201944


(4124, 7)

In [4]:
display(df_products_usd.head(2))
df_products_usd.shape

,category_id,product_id,product_name,price_ars,price_usd
0,15,138,Geishas Tropical,12500.0,8.38
1,33,141,Harumaki,6600.0,4.42


(110, 5)

In [5]:
display(df_orders_usd .head(2))
df_orders_usd.shape


,order_id,client_id,date,time,amount_ars,amount_usd,ars_per_usd
0,3786,3.0,2016-09-30,17:38:19,15.76,1.03,15.31
1,3788,7.0,2016-10-01,00:49:29,315.00,20.57,15.31


(44216, 7)

In [6]:
display(df_product_orders_usd .head(2))
df_product_orders_usd .shape

,order_id,item_id,date,time,price_ars,price_usd,qty,total_ars,total_usd,ars_per_usd
0,4783,153,2017-01-14,19:36:53,390.0,24.60,1,390.0,24.60,15.85
1,4783,149,2017-01-14,19:36:54,98.0,6.18,1,98.0,6.18,15.85


(92710, 10)

In [7]:
display(df_rates.head(2))
df_rates.shape


,date,ars_per_usd
0,2016-01-01,13.04
1,2016-01-02,13.04


(3729, 2)

<div style="
    background: linear-gradient(#0c4555);
    padding: 18px 22px;
    border-radius: 10px;
    font-size: 16px;
    font-weight: 500;
    color: #ffffff;
    line-height: 1.6;
    font-family: -apple-system, BlinkMacSystemFont, 'Segoe UI', Roboto, sans-serif;
    border-left: 4px solid #44b5f7;
">
Ventas por año (ARS vs USD), obtengo facturacion y pedidos, “Las ventas en dólares fueron calculadas utilizando el tipo de cambio de cada día, evitando distorsiones por promedios anuales”
</div>

In [8]:
df_orders_usd["year"] = pd.to_datetime(df_orders_usd["date"]).dt.year

ventas_anuales = df_orders_usd.groupby("year").agg({
    "amount_ars": "sum",
    "amount_usd": "sum",
    "order_id": "count"
}).reset_index()

ventas_anuales.rename(columns={"order_id": "orders"}, inplace=True)

In [9]:
pd.options.display.float_format = '{:,.0f}'.format
ventas_anuales.head(11)

,year,amount_ars,amount_usd,orders
0,2016,"248,347","15,968",837
1,2017,"1,558,835","94,109",4329
2,2018,"1,971,728","73,349",4096
3,2019,"2,612,100","54,611",3495
4,2020,"6,273,278","85,606",5355
5,2021,"14,437,196","150,909",8025
6,2022,"18,700,866","145,188",5891
7,2023,"34,419,632","116,943",4442
8,2024,"90,873,644","98,058",3911
9,2025,"117,387,715","94,907",3284


<div style="
    background: linear-gradient(#0c4555);
    padding: 18px 22px;
    border-radius: 10px;
    font-size: 16px;
    font-weight: 500;
    color: #ffffff;
    line-height: 1.6;
    font-family: -apple-system, BlinkMacSystemFont, 'Segoe UI', Roboto, sans-serif;
    border-left: 4px solid #44b5f7;
">
Ordenes por año. Demanda, si suben ventas pero no pedidos → inflación
</div>

In [10]:
orders_por_año = df_orders_usd.groupby("year")["order_id"].count().reset_index()

orders_por_año

,year,order_id
0,2016,837
1,2017,4329
2,2018,4096
3,2019,3495
4,2020,5355
5,2021,8025
6,2022,5891
7,2023,4442
8,2024,3911
9,2025,3284


<div style="
    background: linear-gradient(#0c4555);
    padding: 18px 22px;
    border-radius: 10px;
    font-size: 16px;
    font-weight: 500;
    color: #ffffff;
    line-height: 1.6;
    font-family: -apple-system, BlinkMacSystemFont, 'Segoe UI', Roboto, sans-serif;
    border-left: 4px solid #44b5f7;
">
TICKET PROMEDIO, demuestra si sube precio o consumo
</div>

In [11]:
ventas_anuales["ticket_ars"] = ventas_anuales["amount_ars"] / ventas_anuales["orders"]
ventas_anuales["ticket_usd"] = ventas_anuales["amount_usd"] / ventas_anuales["orders"]

ventas_anuales

,year,amount_ars,amount_usd,orders,ticket_ars,ticket_usd
0,2016,"248,347","15,968",837,297,19
1,2017,"1,558,835","94,109",4329,360,22
2,2018,"1,971,728","73,349",4096,481,18
3,2019,"2,612,100","54,611",3495,747,16
4,2020,"6,273,278","85,606",5355,"1,171",16
5,2021,"14,437,196","150,909",8025,"1,799",19
6,2022,"18,700,866","145,188",5891,"3,174",25
7,2023,"34,419,632","116,943",4442,"7,749",26
8,2024,"90,873,644","98,058",3911,"23,235",25
9,2025,"117,387,715","94,907",3284,"35,745",29


<div style="
    background: linear-gradient(#0c4555);
    padding: 18px 22px;
    border-radius: 10px;
    font-size: 16px;
    font-weight: 500;
    color: #ffffff;
    line-height: 1.6;
    font-family: -apple-system, BlinkMacSystemFont, 'Segoe UI', Roboto, sans-serif;
    border-left: 4px solid #44b5f7;
">
DELIVERY vs TAKEAWAY
</div>

In [12]:
df_orders_usd["tipo"] = df_orders_usd["client_id"].apply(
    lambda x: "takeaway" if pd.isna(x) or x == 0 else "delivery"
)

delivery_year = df_orders_usd.groupby(["year", "tipo"])["order_id"].count().reset_index()

delivery_year.head()

,year,tipo,order_id
0,2016,delivery,298
1,2016,takeaway,539
2,2017,delivery,1924
3,2017,takeaway,2405
4,2018,delivery,1602


<div style="
    background: linear-gradient(#0c4555);
    padding: 18px 22px;
    border-radius: 10px;
    font-size: 16px;
    font-weight: 500;
    color: #ffffff;
    line-height: 1.6;
    font-family: -apple-system, BlinkMacSystemFont, 'Segoe UI', Roboto, sans-serif;
    border-left: 4px solid #44b5f7;
">
PRODUCTOS TOP
</div>

In [13]:
# por dinero/rentabilidad

top_products = df_product_orders_usd.groupby("item_id").agg({
    "total_usd": "sum",
    "qty": "sum"
}).reset_index()

top_products = top_products.sort_values(by="total_usd", ascending=False).head(10)

# merge

top_products = top_products.merge(
    df_products_usd,
    left_on="item_id",
    right_on="product_id",
    how="left"
)

top_products



,item_id,total_usd,qty,category_id,product_id,product_name,price_ars,price_usd
0,147,"107,099",5453,17,147,Himuro x30 SR,"46,500",31
1,148,"93,887",3260,17,148,Minoru x50 SR,"71,200",48
2,153,"74,341",2825,18,153,Minoru x50 V,"57,800",39
3,211,"72,222",3075,24,211,Himuro x30 Premium,"48,900",33
4,152,"68,494",3998,18,152,Himuro x30 V,"38,700",26
5,212,"45,333",1247,24,212,Minoru x50 Premium,"75,600",51
6,213,"40,345",800,24,213,Dai 75 Premium,"94,400",63
7,183,"33,710",835,17,183,Dai Minoru SR,"89,800",60
8,198,"33,638",3798,23,198,Salad Clasica,"16,900",11
9,146,"32,425",2343,17,146,Sakurai x20 SR,"33,900",23


In [14]:
top_products.to_csv("top_products.csv", index=False)

In [15]:
top_products_cantidad = df_product_orders_usd.groupby("item_id").agg({
    "total_usd": "sum",
    "qty": "sum"
}).reset_index()

top_products_cantidad = top_products_cantidad.sort_values(by="qty", ascending=False).head(10)

top_products_cantidad = top_products_cantidad.merge(
    df_products_usd,
    left_on="item_id",
    right_on="product_id",
    how="left"
)

top_products_cantidad

,item_id,total_usd,qty,category_id,product_id,product_name,price_ars,price_usd
0,158,"4,752",8559,20,158,SALSA TERIYAKI,600,0
1,147,"107,099",5453,17,147,Himuro x30 SR,"46,500",31
2,152,"68,494",3998,18,152,Himuro x30 V,"38,700",26
3,198,"33,638",3798,23,198,Salad Clasica,"16,900",11
4,141,"13,101",3635,33,141,Harumaki,"6,600",4
5,148,"93,887",3260,17,148,Minoru x50 SR,"71,200",48
6,211,"72,222",3075,24,211,Himuro x30 Premium,"48,900",33
7,153,"74,341",2825,18,153,Minoru x50 V,"57,800",39
8,145,"26,701",2352,17,145,Hideki x16 SR,"29,800",20
9,146,"32,425",2343,17,146,Sakurai x20 SR,"33,900",23


In [16]:
top_products_cantidad.to_csv("top_products_cantidad.csv", index=False)

<div style="
    background: linear-gradient(#0c4555);
    padding: 18px 22px;
    border-radius: 10px;
    font-size: 16px;
    font-weight: 500;
    color: #ffffff;
    line-height: 1.6;
    font-family: -apple-system, BlinkMacSystemFont, 'Segoe UI', Roboto, sans-serif;
    border-left: 4px solid #44b5f7;
">
PARETO ( en grafico: x: productos y: acumulado %)
</div>

In [17]:
productos_unicos = df_product_orders_usd.groupby("item_id").agg({
    "total_usd": "sum",
    "qty": "sum"
}).reset_index()

productos_unicos = productos_unicos.sort_values(by="total_usd", ascending=False)

productos_unicos["pct"] = productos_unicos["total_usd"] / productos_unicos["total_usd"].sum() * 100

productos_unicos["cum_pct"] = productos_unicos["pct"].cumsum()

productos_unicos = productos_unicos.merge(
    df_products_usd[["product_id", "product_name"]],
    left_on="item_id",
    right_on="product_id",
    how="left"
)

productos_unicos = productos_unicos[[
    "product_name",
    "total_usd",
    "qty",
    "pct",
    "cum_pct"
]]

productos_unicos["pct"] = productos_unicos["pct"].round(2)
productos_unicos["cum_pct"] = productos_unicos["cum_pct"].round(2)

productos_unicos.shape


(247, 5)

In [18]:
productos_unicos = productos_unicos[~productos_unicos["product_name"].str.contains("1/2", na=False)]

In [19]:
productos_unicos.dropna().reset_index(drop=True)

,product_name,total_usd,qty,pct,cum_pct
0,Himuro x30 SR,"107,099",5453,11,11
1,Minoru x50 SR,"93,887",3260,9,20
2,Minoru x50 V,"74,341",2825,7,27
3,Himuro x30 Premium,"72,222",3075,7,34
4,Himuro x30 V,"68,494",3998,7,41
...,...,...,...,...,...
73,GUACAMOLE,10,12,0,100
74,Salsa Soja light,7,14,0,100
75,salsa teriyaki fg,0,52,0,100
76,1/5 hots free,0,11,0,100


In [20]:
productos_unicos.head(10).to_csv("productos_pareto.csv", index=False)

<div style="
    background: linear-gradient(#0c4555);
    padding: 18px 22px;
    border-radius: 10px;
    font-size: 16px;
    font-weight: 500;
    color: #ffffff;
    line-height: 1.6;
    font-family: -apple-system, BlinkMacSystemFont, 'Segoe UI', Roboto, sans-serif;
    border-left: 4px solid #44b5f7;
">
CLIENTES, tipo de cliente si es recurrente o ocacional
</div>

In [21]:
total_negocio = df_orders_usd["amount_usd"].sum()

df_clientes_reales = df_orders_usd[
    (df_orders_usd["client_id"] != 0) & 
    (df_orders_usd["client_id"].notna())
]

clientes = df_clientes_reales.groupby("client_id").agg({
    "order_id": "count",
    "amount_usd": "sum"
}).reset_index()

clientes.rename(columns={
    "order_id": "num_orders",
    "amount_usd": "total_spent"
}, inplace=True)

def clasificar_cliente(x):
    if x == 1:
        return "único"
    elif x <= 4:
        return "ocasional"
    elif x <= 10:
        return "frecuente"
    else:
        return "vip"

clientes["tipo_cliente"] = clientes["num_orders"].apply(clasificar_cliente)

clientes["pct_orders"] = clientes["num_orders"] / clientes["num_orders"].sum() * 100

clientes["pct_spent"] = clientes["total_spent"] / total_negocio * 100

clientes["pct_orders"] = clientes["pct_orders"].round(2)
clientes["pct_spent"] = clientes["pct_spent"].round(2)

clientes = clientes.sort_values(by="num_orders", ascending=False)

clientes.head(10)


,client_id,num_orders,total_spent,tipo_cliente,pct_orders,pct_spent
1865,"1,981",143,"2,906",vip,1,0
1792,"1,908",134,"3,178",vip,1,0
1592,"1,704",132,933,vip,1,0
65,122,128,"1,868",vip,1,0
17,35,126,"3,979",vip,1,0
1280,"1,382",125,"2,763",vip,1,0
49,88,114,"1,410",vip,1,0
1839,"1,955",111,"1,495",vip,1,0
348,434,107,"2,402",vip,1,0
22,45,105,"1,715",vip,1,0


<div style="
    background: linear-gradient(#0c4555);
    padding: 18px 22px;
    border-radius: 10px;
    font-size: 16px;
    font-weight: 500;
    color: #ffffff;
    line-height: 1.6;
    font-family: -apple-system, BlinkMacSystemFont, 'Segoe UI', Roboto, sans-serif;
    border-left: 4px solid #44b5f7;
">
CLIENTES, por año
</div>

In [22]:
clientes_unicos = clientes[clientes["num_orders"] == 1]

clientes_unicos_detalle = df_clientes_reales.merge(
    clientes_unicos[["client_id"]],
    on="client_id",
    how="inner"
)

clientes_unicos_detalle = df_clientes_reales.merge(
    clientes_unicos[["client_id"]],
    on="client_id",
    how="inner"
)

clientes_unicos = clientes[clientes["num_orders"] == 1]

clientes_unicos_detalle = df_clientes_reales.merge(
    clientes_unicos[["client_id"]],
    on="client_id",
    how="inner"
)

nuevos_por_anio = (
    clientes_unicos_detalle
    .groupby("year")["client_id"]
    .nunique()
    .reset_index(name="clientes_unicos")
    .sort_values("year")
)

nuevos_por_anio

,year,clientes_unicos
0,2016,42
1,2017,217
2,2018,166
3,2019,120
4,2020,244
5,2021,301
6,2022,196
7,2023,130
8,2024,220
9,2025,370


In [23]:
fig = px.bar(
    nuevos_por_anio,
    x="year",
    y="clientes_unicos",
    text="clientes_unicos",
    title="Clientes de una sola compra por año",
    labels={
        "year": "Año",
        "clientes_unicos": "Clientes (1 compra)"
    },
    width=1100,
    height=500
)

fig.update_traces(textposition="outside")

fig.show()

In [24]:
total_clientes_anio = (
    df_clientes_reales
    .groupby("year")["client_id"]
    .nunique()
    .reset_index(name="total_clientes")
)

nuevos_por_anio = nuevos_por_anio.merge(total_clientes_anio, on="year")

nuevos_por_anio["pct_unicos"] = (
    nuevos_por_anio["clientes_unicos"] / nuevos_por_anio["total_clientes"] * 100
).round(2)

In [25]:
clientes["num_orders"].describe()

count   4,180
mean        5
std        10
min         1
25%         1
50%         2
75%         4
max       143
Name: num_orders, dtype: float64

In [26]:
clientes["total_spent"].sum()
df_orders_usd["amount_usd"].sum()

np.float64(947202.75)

<div style="
    background: linear-gradient(#0c4555);
    padding: 18px 22px;
    border-radius: 10px;
    font-size: 16px;
    font-weight: 500;
    color: #ffffff;
    line-height: 1.6;
    font-family: -apple-system, BlinkMacSystemFont, 'Segoe UI', Roboto, sans-serif;
    border-left: 4px solid #44b5f7;
">
Obtener df_order_clients y exportar datos_barrios para Mapa Coropletico
</div>

In [27]:
df_order_clients = df_orders_usd.merge(df_clients, on="client_id")

resumen = df_order_clients.groupby("barrio").agg(
    df_clientes=("client_id", "nunique"),
    df_orders=("order_id", "count")
).reset_index()

<div style="
    background: linear-gradient(#0c4555);
    padding: 18px 22px;
    border-radius: 10px;
    font-size: 16px;
    font-weight: 500;
    color: #ffffff;
    line-height: 1.6;
    font-family: -apple-system, BlinkMacSystemFont, 'Segoe UI', Roboto, sans-serif;
    border-left: 4px solid #44b5f7;
">

INFLACIÓN ANUAL ARGENTINA (Se utilizó la inflación anual publicada por INDEC como referencia macroeconómica)

1. growth_ars - crecimiento anual en pesos
“cuánto crecieron tus ventas en ARS respecto al año anterior”

2. growth_usd - crecimiento real 
“cómo creció el negocio en términos reales”

3. inflation_pct - inflación del país ese año

4. growth_vs_inflation - (growth_ars - inflation_pct)
“¿crecí por encima o por debajo de la inflación?”

5. INTERPRETACIÓN

Si da positivo:  
creciste en términos reales  

Si da negativo:  
creciste menos que la inflación  
perdiste poder real  

“Se excluyeron los años con datos incompletos para asegurar comparabilidad en el análisis”


</div>

In [28]:
inflacion = pd.DataFrame({
    "year": [2016,2017,2018,2019,2020,2021,2022,2023,2024,2025],
    "inflation_pct": [40.9,24.8,47.6,53.8,36.1,50.9,94.8,211.4,117.76,31.55]
})

df_inflacion = ventas_anuales.merge(inflacion, on="year", how="left")

In [29]:
df_inflacion = df_inflacion[df_inflacion["year"] > 2016]
df_inflacion = df_inflacion[df_inflacion["year"] < 2026]

In [30]:
df_inflacion["growth_ars"] = df_inflacion["amount_ars"].pct_change() * 100
df_inflacion["growth_usd"] = df_inflacion["amount_usd"].pct_change() * 100

In [31]:
df_inflacion["growth_vs_inflation"] = df_inflacion["growth_ars"] - df_inflacion["inflation_pct"]

In [32]:
tipo_cambio = df_orders_usd.groupby("year")["ars_per_usd"].mean().reset_index()

tipo_cambio

,year,ars_per_usd
0,2016,16
1,2017,17
2,2018,28
3,2019,48
4,2020,73
5,2021,95
6,2022,127
7,2023,302
8,2024,915
9,2025,"1,247"


<div style="
    background: linear-gradient(#0c4555);
    padding: 18px 22px;
    border-radius: 10px;
    font-size: 16px;
    font-weight: 500;
    color: #ffffff;
    line-height: 1.6;
    font-family: -apple-system, BlinkMacSystemFont, 'Segoe UI', Roboto, sans-serif;
    border-left: 4px solid #44b5f7;
">
¿Cómo evolucionó el volumen total de ventas del negocio a lo largo de los años?
</div>

In [33]:
ventas_anuales = ventas_anuales[
    (ventas_anuales["year"] > 2016) & 
    (ventas_anuales["year"] < 2026)
]

In [34]:
fig = px.line(
    ventas_anuales,
    x="year",
    y="amount_ars",
    title="Evolución de ventas en pesos (ARS)",
    markers=True,
    labels={
        "year": "Año",
        "amount_ars": "Ventas (ARS)"
    },
    width=1500,   
    height=500
)

fig.show()


<div style="
    background: linear-gradient(#b35a00);
    padding: 18px 22px;
    border-radius: 10px;
    font-size: 16px;
    font-weight: 500;
    color: #ffffff;
    line-height: 1.6;
    font-family: -apple-system, BlinkMacSystemFont, 'Segoe UI', Roboto, sans-serif;
    border-left: 4px solid #f7af13;
">
Observaciones

El negocio muestra un crecimiento sostenido en ventas en pesos a lo largo de los años.  
Se observa una aceleración significativa a partir de 2023, con un salto muy marcado en 2024 y 2025.    
</div>

<div style="
    background: linear-gradient(#0c4555);
    padding: 18px 22px;
    border-radius: 10px;
    font-size: 16px;
    font-weight: 500;
    color: #ffffff;
    line-height: 1.6;
    font-family: -apple-system, BlinkMacSystemFont, 'Segoe UI', Roboto, sans-serif;
    border-left: 4px solid #44b5f7;
">
¿La variacion en pesos muestra una tendencia clara de crecimiento del negocio?
</div>

In [35]:
fig = px.bar(
    df_inflacion,
    x="year",
    y="growth_ars",
    title="Variacion anual en pesos (ARS)",
    labels={
        "year": "Año",
        "growth_ars": "Variacion (%)"
    },
    width=1000,   
    height=600
)

fig.show()


<div style="
    background: linear-gradient(#b35a00);
    padding: 18px 22px;
    border-radius: 10px;
    font-size: 16px;
    font-weight: 500;
    color: #ffffff;
    line-height: 1.6;
    font-family: -apple-system, BlinkMacSystemFont, 'Segoe UI', Roboto, sans-serif;
    border-left: 4px solid #f7af13;
">
Observaciones

El crecimiento anual en pesos muestra un comportamiento muy irregular a lo largo del tiempo.  
Se observa un crecimiento extremadamente alto en algunos años, especialmente en 2020 y 2024.  
Existen años con crecimiento moderado e incluso desaceleración, como 2022 y 2025.  
El crecimiento no es estable ni constante, sino que presenta fuertes variaciones año a año.  
Estos picos de crecimiento pueden estar influenciados por aumentos de precios más que por un crecimiento real del negocio.     
El crecimiento en pesos no es consistente y puede estar distorsionado por factores externos como la inflación.
</div>

<div style="
    background: linear-gradient(#0c4555);
    padding: 18px 22px;
    border-radius: 10px;
    font-size: 16px;
    font-weight: 500;
    color: #ffffff;
    line-height: 1.6;
    font-family: -apple-system, BlinkMacSystemFont, 'Segoe UI', Roboto, sans-serif;
    border-left: 4px solid #44b5f7;
">
¿Qué ocurre cuando analizamos esas mismas ventas en dólares?
</div>

In [36]:
fig = px.line(
    df_inflacion,
    x="year",
    y="amount_usd",
    title="Evolución de ventas en dólares (USD)",
    markers=True,
    labels={
        "year": "Año",
        "amount_usd": "Ventas (USD)"
    },
    width=1500,   
    height=500
)

fig.show()

In [37]:
fig = px.bar(
    df_inflacion,
    x="year",
    y="growth_usd",
    title="Variacion anual en dólares (USD)",
    labels={
        "year": "Año",
        "growth_usd": "Variacion (%)"
    },
    width=1000,   
    height=600
)

fig.show()

<div style="
    background: linear-gradient(#b35a00);
    padding: 18px 22px;
    border-radius: 10px;
    font-size: 16px;
    font-weight: 500;
    color: #ffffff;
    line-height: 1.6;
    font-family: -apple-system, BlinkMacSystemFont, 'Segoe UI', Roboto, sans-serif;
    border-left: 4px solid #f7af13;
">
Observaciones  

El crecimiento anual en dólares presenta un comportamiento altamente volátil a lo largo del tiempo.  
Se observan caídas en los primeros años (2018 y 2019), seguidas de un fuerte crecimiento en 2020 y 2021.  
A partir de 2022, el negocio vuelve a mostrar caídas o estancamiento en términos reales.  
En los últimos años, no se evidencia un crecimiento sostenido en dólares, sino fluctuaciones con tendencia a la baja.  

Insight  

El negocio no muestra un crecimiento real consistente, alternando entre años de expansión y contracción en términos de dólares.  
El crecimiento del negocio no es sostenido en términos reales, lo que indica que el aumento en pesos puede estar impulsado principalmente por inflación.  

ARS → crece fuerte  
USD → sube y baja  
growth USD → confirma inestabilidad  
</div>

<div style="
    background: linear-gradient(#0c4555);
    padding: 18px 22px;
    border-radius: 10px;
    font-size: 16px;
    font-weight: 500;
    color: #ffffff;
    line-height: 1.6;
    font-family: -apple-system, BlinkMacSystemFont, 'Segoe UI', Roboto, sans-serif;
    border-left: 4px solid #44b5f7;
">
Comparamos Variacion % USD con ARS
</div>

In [38]:
fig = px.bar(
    df_inflacion,
    x="year",
    y=["growth_ars", "growth_usd"],
    barmode="group",
    title="Variacion anual: ARS vs USD",
    labels={
        "value": "Variacion (%)",
        "year": "Año",
        "variable": "Tipo"
    },
    width=1000,
    height=600
)

fig.show()

In [39]:
df_inflacion = df_inflacion.sort_values("year")

df_inflacion["growth_ars"] = df_inflacion["amount_ars"].pct_change() * 100
df_inflacion["growth_usd"] = df_inflacion["amount_usd"].pct_change() * 100

df_growth = df_inflacion[["year", "growth_ars", "growth_usd"]]

df_growth.to_csv("growth_ars_vs_usd.csv", index=False)

df_growth.head()

,year,growth_ars,growth_usd
1,2017,NaN,NaN
2,2018,26,-22
3,2019,32,-26
4,2020,140,57
5,2021,130,76


<div style="
    background: linear-gradient(#0c4555);
    padding: 18px 22px;
    border-radius: 10px;
    font-size: 16px;
    font-weight: 500;
    color: #ffffff;
    line-height: 1.6;
    font-family: -apple-system, BlinkMacSystemFont, 'Segoe UI', Roboto, sans-serif;
    border-left: 4px solid #44b5f7;
">
Comparamos Variacion % con Inflación
</div>

In [40]:
px.bar(
    df_inflacion,
    x="year",
    y="growth_vs_inflation",
    title="Crecimiento real del negocio (ajustado por inflación)",
    labels={
        "growth_vs_inflation": "Crecimiento real (%)",
        "year": "Año"
    },
    width=1000,   
    height=600
)

<div style="
    background: linear-gradient(#0c4555);
    padding: 18px 22px;
    border-radius: 10px;
    font-size: 16px;
    font-weight: 500;
    color: #ffffff;
    line-height: 1.6;
    font-family: -apple-system, BlinkMacSystemFont, 'Segoe UI', Roboto, sans-serif;
    border-left: 4px solid #44b5f7;
">
¿Cómo evolucionó la cantidad de pedidos a lo largo del tiempo?
</div>

In [41]:
fig = px.line(
    df_inflacion,
    x="year",
    y="orders",
    title="Evolución de pedidos",
    markers=True,
    labels={
        "year": "Año",
        "orders": "Cantidad de pedidos"
    },
    width=1500,   
    height=500
)

fig.show()

In [42]:
df_inflacion = df_inflacion.sort_values("year")

df_orders = df_inflacion[["year", "orders"]]

df_orders.to_csv("orders_evolution.csv", index=False)

df_inflacion["growth_orders"] = df_inflacion["orders"].pct_change() * 100

df_growth = df_inflacion[["year", "growth_orders"]]

df_growth.to_csv("orders_growth.csv", index=False)

In [43]:
df_orders_usd["date"] = pd.to_datetime(df_orders_usd["date"])

df_orders_usd["year"] = df_orders_usd["date"].dt.year
df_orders_usd["month"] = df_orders_usd["date"].dt.month

df_19_20 = df_orders_usd[df_orders_usd["year"].isin([2019, 2020])]

df_monthly = df_19_20.groupby(["year", "month"])[["amount_ars", "amount_usd"]].mean().reset_index()

df_monthly = df_monthly.rename(columns={
    "amount_ars": "ticket_ars",
    "amount_usd": "ticket_usd"
})

df_monthly = df_monthly.sort_values(["year", "month"])
df_monthly.to_csv("ticket_mensual_2019_2020.csv", index=False)

df_monthly.head()

,year,month,ticket_ars,ticket_usd
0,2019,1,575,15
1,2019,2,608,16
2,2019,3,608,15
3,2019,4,676,16
4,2019,5,650,14


In [44]:
df_orders_usd["date"] = pd.to_datetime(df_orders_usd["date"])

df_orders_usd["year"] = df_orders_usd["date"].dt.year
df_orders_usd["month"] = df_orders_usd["date"].dt.month

df_19_20 = df_orders_usd[df_orders_usd["year"].isin([2019, 2020])]

df_orders_usd_monthly = df_19_20.groupby(["year", "month"])["order_id"].count().reset_index()

df_orders_usd_monthly = df_orders_usd_monthly.rename(columns={
    "order_id": "orders"
})

df_orders_usd_monthly = df_orders_usd_monthly.sort_values(["year", "month"])

df_orders_usd_monthly.to_csv("orders_mensual_2019_2020.csv", index=False)

df_orders_usd_monthly.head()

,year,month,orders
0,2019,1,316
1,2019,2,306
2,2019,3,331
3,2019,4,316
4,2019,5,259


In [45]:
df_orders_usd["date"] = pd.to_datetime(df_orders_usd["date"])

df_orders_usd["year"] = df_orders_usd["date"].dt.year
df_orders_usd["month"] = df_orders_usd["date"].dt.month

df_19_20 = df_orders_usd[df_orders_usd["year"].isin([2019, 2020])]

df_grouped = df_19_20.groupby(["year", "month"])["order_id"].count().reset_index()

df_pivot = df_grouped.pivot(
    index="month",
    columns="year",
    values="order_id"
)

df_pivot = df_pivot.rename(columns={
    2019: "orders_2019",
    2020: "orders_2020"
})

df_pivot = df_pivot.reset_index()

df_pivot = df_pivot.sort_values("month")

df_pivot.to_csv("orders_mensual_2019_vs_2020.csv", index=False)

df_pivot.head()

year,month,orders_2019,orders_2020
0,1,316,256
1,2,306,283
2,3,331,172
3,4,316,199
4,5,259,436


In [46]:
fig = px.bar(
    df_orders_usd_monthly,
    x="month",
    y="orders",
    color="year",
    barmode="group",
    title="Pedidos mensuales: 2019 vs 2020",
    width=1500,
    height=500
)

fig.show()

In [47]:
df_orders_usd["date"] = pd.to_datetime(df_orders_usd["date"])
df_orders_usd["year"] = df_orders_usd["date"].dt.year

df_ticket = df_orders_usd.groupby("year")[["amount_ars", "amount_usd"]].mean().reset_index()

df_ticket["growth_ticket_ars"] = df_ticket["amount_ars"].pct_change() * 100
df_ticket["growth_ticket_usd"] = df_ticket["amount_usd"].pct_change() * 100

df_ticket

df_ticket.to_csv("df_porcentaje_ticketmedio.csv", index=False)

<div style="
    background: linear-gradient(#b35a00);
    padding: 18px 22px;
    border-radius: 10px;
    font-size: 16px;
    font-weight: 500;
    color: #ffffff;
    line-height: 1.6;
    font-family: -apple-system, BlinkMacSystemFont, 'Segoe UI', Roboto, sans-serif;
    border-left: 4px solid #f7af13;
">
Observaciones  

La cantidad de pedidos no muestra un crecimiento sostenido a lo largo del tiempo.  
Se observa una caída inicial hasta 2019, seguida de un fuerte crecimiento en 2020 y 2021.  
A partir de 2021, los pedidos entran en una tendencia descendente clara y sostenida.  

El negocio no está creciendo en volumen de pedidos, sino que presenta una caída sostenida en la demanda en los últimos años.  
Mientras las ventas en pesos crecen, la cantidad de pedidos disminuye, lo que sugiere que el crecimiento puede estar impulsado por precios y no por mayor demanda.  
</div>

<div style="
    background: linear-gradient(#0c4555);
    padding: 18px 22px;
    border-radius: 10px;
    font-size: 16px;
    font-weight: 500;
    color: #ffffff;
    line-height: 1.6;
    font-family: -apple-system, BlinkMacSystemFont, 'Segoe UI', Roboto, sans-serif;
    border-left: 4px solid #44b5f7;
">
¿Cómo evolucionó el ticket promedio en pesos y en dólares?
</div>

In [48]:
px.line(
    df_inflacion,
    x="year",
    y="ticket_ars",
    title="Ticket promedio en pesos (ARS)",
    markers=True,
    width=1500,   
    height=500
)


In [49]:
px.line(
    df_inflacion,
    x="year",
    y="ticket_usd",
    title="Ticket promedio en dólares (USD)",
    markers=True,
    width=1500,   
    height=500
)

In [50]:
df_inflacion

,year,amount_ars,amount_usd,orders,ticket_ars,ticket_usd,inflation_pct,growth_ars,growth_usd,growth_vs_inflation,growth_orders
1,2017,"1,558,835","94,109",4329,360,22,25,NaN,NaN,NaN,NaN
2,2018,"1,971,728","73,349",4096,481,18,48,26,-22,-21,-5
3,2019,"2,612,100","54,611",3495,747,16,54,32,-26,-21,-15
4,2020,"6,273,278","85,606",5355,"1,171",16,36,140,57,104,53
5,2021,"14,437,196","150,909",8025,"1,799",19,51,130,76,79,50
6,2022,"18,700,866","145,188",5891,"3,174",25,95,30,-4,-65,-27
7,2023,"34,419,632","116,943",4442,"7,749",26,211,84,-19,-127,-25
8,2024,"90,873,644","98,058",3911,"23,235",25,118,164,-16,46,-12
9,2025,"117,387,715","94,907",3284,"35,745",29,32,29,-3,-2,-16


<div style="
    background: linear-gradient(#b35a00);
    padding: 18px 22px;
    border-radius: 10px;
    font-size: 16px;
    font-weight: 500;
    color: #ffffff;
    line-height: 1.6;
    font-family: -apple-system, BlinkMacSystemFont, 'Segoe UI', Roboto, sans-serif;
    border-left: 4px solid #f7af13;
">
Observaciones  

El ticket promedio en pesos muestra un crecimiento muy fuerte a lo largo del tiempo, especialmente a partir de 2022.  
En contraste, el ticket promedio en dólares presenta un crecimiento mucho más moderado y estable.  
Se observa una caída inicial hasta 2019, seguida de una recuperación progresiva en los años posteriores.  
En los últimos años, el ticket en dólares crece ligeramente, pero sin cambios drásticos.  
</div>

<div style="
    background: linear-gradient(#0c4555);
    padding: 18px 22px;
    border-radius: 10px;
    font-size: 16px;
    font-weight: 500;
    color: #ffffff;
    line-height: 1.6;
    font-family: -apple-system, BlinkMacSystemFont, 'Segoe UI', Roboto, sans-serif;
    border-left: 4px solid #44b5f7;
">
¿Cómo se distribuyen los pedidos entre delivery y takeaway?
</div>

In [51]:
df_orders_usd["tipo"] = df_orders_usd["client_id"].apply(
    lambda x: "delivery" if pd.notnull(x) and x != 0 else "takeaway"
)

canal = df_orders_usd.groupby("tipo")["order_id"].count().reset_index()

fig = px.pie(
    canal,
    names="tipo",
    values="order_id",
    title="Distribución de pedidos: Delivery vs Takeaway",
    width=1000,   
    height=600
)

fig.show()

<div style="
    background: linear-gradient(#b35a00);
    padding: 18px 22px;
    border-radius: 10px;
    font-size: 16px;
    font-weight: 500;
    color: #ffffff;
    line-height: 1.6;
    font-family: -apple-system, BlinkMacSystemFont, 'Segoe UI', Roboto, sans-serif;
    border-left: 4px solid #f7af13;
">
Observaciones

El canal takeaway representa la mayor parte de los pedidos, con aproximadamente un 56.6% del total.  
El delivery concentra alrededor del 43.4% de los pedidos.  
La distribución muestra una ligera predominancia del canal takeaway sobre el delivery.  

Insight  
El negocio depende en mayor medida del canal takeaway, aunque el delivery también representa una parte significativa de la demanda.  
El takeaway es el canal principal del negocio, lo que sugiere una fuerte dependencia del consumo directo en el local o retiro.

</div>

<div style="
    background: linear-gradient(#0c4555);
    padding: 18px 22px;
    border-radius: 10px;
    font-size: 16px;
    font-weight: 500;
    color: #ffffff;
    line-height: 1.6;
    font-family: -apple-system, BlinkMacSystemFont, 'Segoe UI', Roboto, sans-serif;
    border-left: 4px solid #44b5f7;
">
¿Cómo evolucuinan estos canales a lo largo del tiempo?
</div>

In [52]:
canal_year = df_orders_usd.groupby(["year", "tipo"])["order_id"].count().reset_index()

fig = px.line(
    canal_year,
    x="year",
    y="order_id",
    color="tipo",
    title="Evolución de pedidos por canal",
    markers=True,
    width=1500,   
    height=500
)

fig.show()

In [53]:
canal_year = df_orders_usd.groupby(["year", "tipo"])["order_id"].count().reset_index()

df_canal = canal_year.pivot(
    index="year",
    columns="tipo",
    values="order_id"
)

df_canal = df_canal.reset_index()

df_canal.to_csv("pedidos_por_canal.csv", index=False)

<div style="
    background: linear-gradient(#b35a00);
    padding: 18px 22px;
    border-radius: 10px;
    font-size: 16px;
    font-weight: 500;
    color: #ffffff;
    line-height: 1.6;
    font-family: -apple-system, BlinkMacSystemFont, 'Segoe UI', Roboto, sans-serif;
    border-left: 4px solid #f7af13;
">
Observaciones  
  
Historicamente el delivery se mantiene por debajo del takeaway salvo en el ultimo año, se desprende mas en epocas de alta inflación al rededor del 2018/2019 y el 2023 
</div>

<div style="
    background: linear-gradient(#0c4555);
    padding: 18px 22px;
    border-radius: 10px;
    font-size: 16px;
    font-weight: 500;
    color: #ffffff;
    line-height: 1.6;
    font-family: -apple-system, BlinkMacSystemFont, 'Segoe UI', Roboto, sans-serif;
    border-left: 4px solid #44b5f7;
">
¿Cómo se distribuyen las ventas por barrio?
</div>

In [54]:
df_orders_geo = df_orders_usd.merge(
    df_clients[["client_id", "barrio"]],
    on="client_id",
    how="left"
)

geo = df_orders_geo.groupby("barrio").agg({
    "amount_usd": "sum",
    "client_id": "nunique"
}).reset_index()

geo.rename(columns={
    "amount_usd": "ventas_usd",
    "client_id": "clientes"
}, inplace=True)

In [55]:
geo["pct_ventas"] = geo["ventas_usd"] / geo["ventas_usd"].sum() * 100

geo["pct_ventas"] = geo["pct_ventas"].round(2)

geo.sort_values("pct_ventas", ascending=False)

,barrio,ventas_usd,clientes,pct_ventas
1,beccar,"232,408",2332,58
8,san isidro,"84,485",840,21
6,martinez,"28,938",304,7
11,victoria,"22,536",286,6
0,acassuso,"20,756",139,5
7,san fernando,"3,508",29,1
10,vicente lopez,"2,954",3,1
3,don torcuato,"1,508",20,0
2,boulogne,"1,124",32,0
5,la lucila,694,21,0


In [56]:
top = geo.sort_values("pct_ventas", ascending=False).head(5)
otros = geo.sort_values("pct_ventas", ascending=False).iloc[5:]

otros_sum = otros["pct_ventas"].sum()

top = pd.concat([
    top,
    pd.DataFrame([{"barrio": "otros", "pct_ventas": otros_sum}])
])

In [57]:
fig = px.pie(
    top,
    names="barrio",
    values="pct_ventas",
    title="Participación de ventas por barrio (%)",
    hole=0.5,
    width=1000,   
    height=600
)

fig.show()

In [58]:
top = geo.sort_values("pct_ventas", ascending=False).head(5)

otros = geo.sort_values("pct_ventas", ascending=False).iloc[5:]
otros_sum = otros["pct_ventas"].sum()

df_ventas_barrio = pd.concat([
    top,
    pd.DataFrame([{"barrio": "otros", "pct_ventas": otros_sum}])
], ignore_index=True)

df_ventas_barrio.to_csv("participacion_ventas_barrios.csv", index=False)

<div style="
    background: linear-gradient(#b35a00);
    padding: 18px 22px;
    border-radius: 10px;
    font-size: 16px;
    font-weight: 500;
    color: #ffffff;
    line-height: 1.6;
    font-family: -apple-system, BlinkMacSystemFont, 'Segoe UI', Roboto, sans-serif;
    border-left: 4px solid #f7af13;
">
Observaciones  

La mayor concentración de clientes se encuentra en Beccar, seguido por San Isidro y Martínez. Esta distribución se refleja también en los ingresos, donde   Beccar lidera ampliamente con más de la mitad de las ventas.  

El negocio presenta una fuerte concentración geográfica, tanto en clientes como en ingresos, dependiendo principalmente de pocos barrios.  
</div>

In [59]:
df = df_orders_usd.merge(
    df_clients[["client_id", "barrio"]],
    on="client_id",
    how="left"
)

df_barrio = df.groupby("barrio", as_index=False)["amount_ars"].sum()

df_barrio = df_barrio.rename(columns={"amount_ars": "ingresos_ars"})

total = df_barrio["ingresos_ars"].sum()

df_barrio["pct_ingresos"] = (df_barrio["ingresos_ars"] / total) * 100

df_barrio = df_barrio.sort_values("pct_ingresos", ascending=False)

df_barrio["pct_ingresos"] = df_barrio["pct_ingresos"].round(2)

In [60]:
df_barrio.to_csv("ingresos_por_barrio.csv", index=False)

<div style="
    background: linear-gradient(#0c4555);
    padding: 18px 22px;
    border-radius: 10px;
    font-size: 16px;
    font-weight: 500;
    color: #ffffff;
    line-height: 1.6;
    font-family: -apple-system, BlinkMacSystemFont, 'Segoe UI', Roboto, sans-serif;
    border-left: 4px solid #44b5f7;
">
¿Cómo se distribuyen los clientes según su frecuencia de compra?

</div>

In [61]:
def grupo(x):
    if x == 1:
        return "1 pedido"
    elif x <= 3:
        return "2-3 pedidos"
    elif x <= 10:
        return "4-10 pedidos"
    elif x <= 30:
        return "11-30 pedidos"
    else:
        return "30+ pedidos"

clientes["grupo"] = clientes["num_orders"].apply(grupo)

dist = clientes["grupo"].value_counts().reset_index()
dist.columns = ["grupo", "num_clientes"]

px.bar(
    dist,
    x="grupo",
    y="num_clientes",
    title="Distribución de clientes por frecuencia de compra",
    labels={
        "grupo": "Frecuencia de compra",
        "num_clientes": "Número de clientes"
    },
    width=1000,   
    height=600
)

In [62]:
map_grupos = {
    "1 pedido": "Nuevos",
    "2-3 pedidos": "Recurrentes",
    "4-10 pedidos": "Frecuentes",
    "11-30 pedidos": "Leales",
    "30+ pedidos": "VIP"
}

dist["etapa"] = dist["grupo"].map(map_grupos)

orden = ["Nuevos", "Recurrentes", "Frecuentes", "Leales", "VIP"]

df_funnel = dist.copy()
df_funnel = df_funnel.groupby("etapa", as_index=False)["num_clientes"].sum()
df_funnel["etapa"] = pd.Categorical(df_funnel["etapa"], categories=orden, ordered=True)
df_funnel = df_funnel.sort_values("etapa")

total = df_funnel["num_clientes"].sum()
df_funnel["pct"] = (df_funnel["num_clientes"] / total * 100).round(2)

In [63]:
df_funnel.to_csv("funnel_clientes.csv", index=False)

<div style="
    background: linear-gradient(#b35a00);
    padding: 18px 22px;
    border-radius: 10px;
    font-size: 16px;
    font-weight: 500;
    color: #ffffff;
    line-height: 1.6;
    font-family: -apple-system, BlinkMacSystemFont, 'Segoe UI', Roboto, sans-serif;
    border-left: 4px solid #f7af13;
">
Observaciones  

La mitad de los clientes realiza una única compra, mientras que solo una pequeña proporción muestra recurrencia 
</div>

<div style="
    background: linear-gradient(#0c4555);
    padding: 18px 22px;
    border-radius: 10px;
    font-size: 16px;
    font-weight: 500;
    color: #ffffff;
    line-height: 1.6;
    font-family: -apple-system, BlinkMacSystemFont, 'Segoe UI', Roboto, sans-serif;
    border-left: 4px solid #44b5f7;
">
Heatmap con top 10 clientes por cantidad de pedidos anuales

</div>

In [64]:

df["date"] = pd.to_datetime(df["date"])
df["year"] = df["date"].dt.year


top_clientes = df.groupby("client_id")["order_id"].nunique().sort_values(ascending=False).head(20).index


df_top = df[df["client_id"].isin(top_clientes)]


df_pedidos = df_top.groupby(["client_id", "year"]).agg({
    "order_id": "nunique"
}).reset_index()

df_pedidos.rename(columns={"order_id": "orders"}, inplace=True)


tabla_top = df_pedidos.pivot(
    index="client_id",
    columns="year",
    values="orders"
)


tabla_top = tabla_top.fillna(0).astype(int)


tabla_top["total"] = tabla_top.sum(axis=1)
tabla_top = tabla_top.sort_values("total", ascending=False)

tabla_top = tabla_top.drop(tabla_top.index[0])
tabla_top

year,2016,2017,2018,2019,2020,2021,2022,2023,2024,2025,2026,total
client_id,,,,,,,,,,,,
"1,981",0,0,0,0,0,62,52,29,0,0,0,143
"1,908",0,0,0,0,3,35,36,28,22,9,1,134
"1,704",0,0,0,0,24,64,33,11,0,0,0,132
122,5,21,9,2,27,45,19,0,0,0,0,128
35,1,50,23,2,9,13,18,6,4,0,0,126
"1,382",0,0,0,0,27,40,26,15,10,6,1,125
88,3,28,21,26,14,12,6,4,0,0,0,114
"1,955",0,0,0,0,2,31,41,26,8,3,0,111
434,0,7,13,17,1,1,8,32,21,7,0,107


<div style="
    background: linear-gradient(#b35a00);
    padding: 18px 22px;
    border-radius: 10px;
    font-size: 16px;
    font-weight: 500;
    color: #ffffff;
    line-height: 1.6;
    font-family: -apple-system, BlinkMacSystemFont, 'Segoe UI', Roboto, sans-serif;
    border-left: 4px solid #f7af13;
">
Observaciones  

Mayor concentracion en años centrales del 2019 al 2023 con datos mas sesgados en los ultimos dos años
</div>